In [3]:
import requests 

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [4]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    "http://localhost:9200",
    verify_certs=False,
    request_timeout=30
)

try:
    print(es.info())
except Exception as e:
    print(f"Error: {e}")


{'name': '99adf594d512', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'AgDLmc5mTJ2OBDLxZ7Ifdg', 'version': {'number': '8.11.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '6f9ff581fbcde658e6f69d6ce03050f060d1fd0c', 'build_date': '2023-11-11T10:05:59.421038163Z', 'build_snapshot': False, 'lucene_version': '9.8.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [7]:
if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)
    print(f"Deleted index: {index_name}")

Deleted index: course-questions


In [8]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = "course-questions"
es.indices.create(index=index_name, body=index_settings)


/tmp/ipykernel_2027/1458963781.py:17: DeprecationWarning: The 'body' parameter is deprecated for the 'create' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.indices.create(index=index_name, body=index_settings)


{'acknowledged': True,
 'shards_acknowledged': True,
 'index': 'course-questions'}

In [9]:
for doc in documents:
    es.index(index=index_name, document=doc)


In [10]:
print(es.count(index=index_name)['count'])


948


In [11]:
query = "How do I execute a command in a running docker container?"

search_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^4", "text"],
                    "type": "best_fields"
                }
            },
        }
    }
}

search_results = es.search(index=index_name, body=search_query)

/tmp/ipykernel_2027/1669299130.py:18: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  search_results = es.search(index=index_name, body=search_query)


In [12]:
search_results['hits']['hits'][0]['_score']

84.050095

In [13]:
query = "How do copy a file to a Docker container?"

search_query = {
    "size": 3,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^4", "text"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "machine-learning-zoomcamp"
                }
            }
        }
    }
}

search_results = es.search(index=index_name, body=search_query)

/tmp/ipykernel_2027/3252389259.py:23: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  search_results = es.search(index=index_name, body=search_query)


In [16]:
context_template = """
Q: {question}
A: {text}
""".strip()

prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [17]:
hits = search_results['hits']['hits']
context_entries = []

# Loop through each hit to grab the inner fields
for hit in hits:
    doc = hit['_source'] # '_source' contains the question and text
    
    # Format using your template from Q5
    formatted_entry = context_template.format(
        question=doc['question'], 
        text=doc['text']
    )
    context_entries.append(formatted_entry)

# Join them by double linebreaks as instructed
context = "\n\n".join(context_entries)

In [18]:
prompt = prompt_template.format(question=query, context=context)

In [19]:
len(prompt)

1446

In [ ]:
import os
import requests
from pathlib import Path
from dotenv import load_dotenv
import tiktoken


current_dir = Path.cwd()
grandparent_dir = current_dir.parent
env_path = grandparent_dir / '.env'
load_dotenv(dotenv_path=env_path)
api_key = os.getenv("OPENROUTER_API_KEY")



Testing key: sk-or-v1-f...


In [40]:
# --- 2. Measure Tokens using tiktoken ---
# 'cl100k_base' is standard for OpenAI GPT models and many completions
encoding = tiktoken.get_encoding("cl100k_base") 
tokens = encoding.encode(prompt)
print(f"Prompt Token Count: {len(tokens)}")

Prompt Token Count: 323


In [41]:
# --- 3. Execute OpenRouter Call ---
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
    "HTTP-Referrer": "http://localhost:3000", 
}

data = {
    "model": "google/gemma-3n-e2b-it:free",
    "messages": [{"role": "user", "content": prompt}]
}

response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers=headers,
    json=data
)

print(f"Status Code: {response.status_code}")

Status Code: 200


In [42]:
if response.status_code == 200:
    try:
        json_data = response.json()
        print("\n✅ Success!")
        
        # Pulling content out of standard array list returned by OpenRouter
        answer = json_data['choices'][0]['message']['content']
        print("\nAI Response:")
        print(answer)
    except Exception as e:
        print(f"❌ Failed to parse JSON: {e}")
else:
    print(f"❌ Request failed with status {response.status_code}")
    print(response.text)



✅ Success!

AI Response:
You can copy files from your local machine into a Docker container using the `docker cp command`. The basic syntax is:

`docker cp /path/to/local/file_or_directory container_id:/path/in/container`

